# Benchmark ODE solver on standard systems

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/benchmarks/ode_solver_benchmark.ipynb)



## Install

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")


## Setup

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / "benchmarks").is_dir():
    repo_root = next(p for p in [repo_root, *repo_root.parents] if (p / "benchmarks").is_dir() and (p / "minilink").is_dir())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import jax

from benchmarks.simulation import (
    DEFAULT_SIMULATION_VARIANTS,
    benchmark_simulation_matrix,
    print_simulation_matrix_benchmark,
)
from benchmarks.systems.basic import make_pendulum
from benchmarks.systems.engine import make_physics_many_spheres
from benchmarks.systems.network import make_dense_network

USE_X64 = False
jax.config.update("jax_enable_x64", USE_X64)

_NRUNS = 5


## Pendulum

In [ ]:
sys = make_pendulum()

sys.compute_trajectory(tf=100.0)
sys.plot_trajectory()

In [ ]:
res = benchmark_simulation_matrix(
    sys,
    case_name="Pendulum",
    variants=DEFAULT_SIMULATION_VARIANTS,
    t0=0.0,
    tf=100.0,
    dt=0.01,
    n_runs=_NRUNS,
)
print_simulation_matrix_benchmark(res)

# Complex diagram

In [ ]:
sys = make_dense_network(num_nodes=50, connections_per_node=5)


sys.plot_diagram()
sys.compute_trajectory(tf=5.0)
sys.plot_trajectory()

In [ ]:
res = benchmark_simulation_matrix(
    sys,
    case_name="Dense network (50 nodes, 5 conn/node)",
    variants=DEFAULT_SIMULATION_VARIANTS,
    t0=0.0,
    tf=5.0,
    dt=0.1,
    n_runs=_NRUNS,
)
print_simulation_matrix_benchmark(res)

## Bouncing spheres

In [ ]:
sys = make_physics_many_spheres(nx=6, ny=4)

sys.compute_trajectory(tf=2.0, show=False)
sys.plot_trajectory()

In [ ]:
sys.animate(renderer="meshcat")


In [ ]:
res = benchmark_simulation_matrix(
    sys,
    case_name="PhysicsManySpheres (24 bodies)",
    variants=DEFAULT_SIMULATION_VARIANTS,
    t0=0.0,
    tf=1.0,
    dt=0.01,
    n_runs=_NRUNS,
)
print_simulation_matrix_benchmark(res)

In [ ]:
print("default backend:", jax.default_backend())